# QLoRA Fine-Tuning on Colab — Support Ticket Triage

Local training (laptop GTX 1660 Ti) hit real thermal throttling under sustained load -- see the project's `docs/build-log.md` for the full story. This notebook runs the same training on Colab's GPU instead: same data, same hyperparameters, same script logic, just different compute.

**Before running:** Runtime menu -> Change runtime type -> T4 GPU (free tier). Then run cells top to bottom.

In [ ]:
!pip install -q transformers peft bitsandbytes trl accelerate datasets

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## Load data

Rebuilds the exact same train/test split used locally (same dataset, same `random_state=42`), rather than requiring a file upload -- this keeps the split identical to what the baseline evaluation already used.

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

DATASET_NAME = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"

ds = load_dataset(DATASET_NAME, split="train")
df = ds.to_pandas()

train_df, test_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df["intent"]
)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

labels = sorted(train_df["intent"].unique().tolist())
print(f"Classes: {len(labels)}")

## Stratified subset for the real run

Measured throughput on this Colab T4 (3 consistent tests): ~1.1 samples/sec, regardless of batch size or gradient checkpointing setting -- that's the real ceiling for this setup, not something to keep tuning. Full dataset x 2 epochs would need ~11 hours, too long for a Colab session. The loss curve during testing dropped from 2.59 to 0.30 in under 3% of one epoch, which suggests this task converges fast and doesn't need the full dataset to show a real improvement.

Plan: 370 examples/class (~10,000 total, still 6x more than local hardware could handle), 1 epoch, ~2.5 hours -- comfortably inside Colab's session limits with real margin against disconnection.

In [ ]:
PER_CLASS = 370

subset_df = (
    train_df.groupby("intent", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), PER_CLASS), random_state=42), include_groups=True)
    .reset_index(drop=True)
)
print(f"Subset size: {len(subset_df)} across {subset_df['intent'].nunique()} classes")

## Build training examples and load model in 4-bit

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_example(row):
    system = (
        "You are a customer support ticket classifier. Given a customer "
        "message, respond with exactly one intent label from this list, and "
        "nothing else:\n" + ", ".join(labels)
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": row["instruction"]},
        {"role": "assistant", "content": row["intent"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

subset_df = subset_df.copy()
subset_df["text"] = subset_df.apply(build_example, axis=1)
train_dataset = Dataset.from_pandas(subset_df[["text"]], preserve_index=False)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Full training run

1 epoch over the 10,000-example subset. At the confirmed ~1.1 samples/sec rate, expect roughly 2.5 hours. Keep the tab open/active -- Colab can disconnect idle sessions.

In [ ]:
from trl import SFTConfig, SFTTrainer

full_config = SFTConfig(
    output_dir="/content/qlora-adapter",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    save_strategy="epoch",
    bf16=True,
    max_length=192,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(model=model, args=full_config, train_dataset=train_dataset, processing_class=tokenizer)
trainer.train()
trainer.save_model("/content/qlora-adapter")
tokenizer.save_pretrained("/content/qlora-adapter")
print("Adapter saved to /content/qlora-adapter")

## Download the adapter

Zips the adapter so it can be pulled back down and registered to the Azure ML Model Registry afterward -- same fallback pattern the project brief specified for Colab-trained models.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/qlora-adapter", "zip", "/content/qlora-adapter")
files.download("/content/qlora-adapter.zip")